In [4]:
import pandas as pd
import pymysql
from typing import Union, List

def fetch_unique_indicators(
    db_info: dict,
    table_name: str = "Korea_company_valuation_ver2"
) -> list:
    """
    DB 테이블에서 indicator 컬럼의 unique 값 리스트 반환
    """

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        sql = f"""
        SELECT DISTINCT indicator
        FROM {table_name}
        ORDER BY indicator
        """
        df = pd.read_sql(sql, conn)
    finally:
        conn.close()

    return df["indicator"].tolist()

def fetch_indicator_pivot(
    db_info: dict,
    indicator: Union[str, List[str]],
    forecast_date: str,
    ticker: str,
    table_name: str = "Korea_company_valuation_ver2"
) -> pd.DataFrame:
    """
    Parameters
    ----------
    db_info : dict
    indicator : str or list[str]
        예: "psr_ETS" 또는 ["psr_ETS", "psr_SARIMA"]
    forecast_date : str
        예: "2025-12-05"
    ticker : str
        예: "A005930"
    table_name : str

    Returns
    -------
    DataFrame
        index   : date
        columns : indicator
        values  : value
    """

    # indicator를 리스트로 통일
    if isinstance(indicator, str):
        indicator = [indicator]

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        indicator_placeholders = ",".join(["%s"] * len(indicator))

        sql = f"""
        SELECT
            date,
            indicator,
            value
        FROM {table_name}
        WHERE forecast_date = %s
          AND ticker = %s
          AND indicator IN ({indicator_placeholders})
        ORDER BY date
        """

        params = [forecast_date, ticker] + indicator
        df = pd.read_sql(sql, conn, params=params)

    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame()

    pivot_df = (
        df.pivot_table(
            index="date",
            columns="indicator",
            values="value",
            aggfunc="last"
        )
        .sort_index()
    )

    return pivot_df


In [3]:
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

indicators = fetch_unique_indicators(
    db_info=db_info,
    table_name="Korea_company_valuation_ver2"
)

print(indicators)

['ensemble_forecast', 'ensemble_valuation', 'exog_var', 'exp_smoothing_forecast', 'forecast_date', 'is_forecast', 'lstm_forecast', 'lstm_valuation', 'matched_ttm_without_exog', 'matched_ttm_with_exog', 'mc_ets', 'mc_lstm', 'mc_prophet', 'mc_sarima_exog', 'mc_sarima_noexog', 'mc_theta', 'prophet_forecast', 'prophet_valuation', 'psr', 'psr_ETS', 'psr_LSTM', 'psr_Prophet', 'psr_SARIMA_exog', 'psr_SARIMA_noexog', 'psr_Theta', 'revenue_ensemble_forecast', 'revenue_ets', 'revenue_ets_ttm', 'revenue_exp_smoothing_forecast', 'revenue_lstm', 'revenue_lstm_forecast', 'revenue_lstm_ttm', 'revenue_prophet', 'revenue_prophet_forecast', 'revenue_prophet_ttm', 'revenue_sarima', 'revenue_sarima_exog', 'revenue_sarima_exog_ttm', 'revenue_sarima_ttm', 'revenue_theta', 'revenue_theta_ttm', 'revenue_without_exog_forecast', 'revenue_with_exog_forecast', 'sarima_forecast', 'sarima_valuation', 'ttm_revenue', 'ttm_revenue_without_exog', 'ttm_revenue_with_exog', 'valuation']


In [19]:
pivot_df = fetch_indicator_pivot(
    db_info=db_info,
    indicator="mc_ets",
    forecast_date="2025-12-04",
    ticker="A000660"
)

print(pivot_df)

indicator              mc_ets
date                         
2026-01-31  625442432786.3044
2026-02-28  557223940439.7587
2026-03-31  607905242543.1605
2026-04-30  620465203846.0746
2026-05-31  631052275011.2697
2026-06-30  659982363990.6958
2026-07-31  665538706213.0454
2026-08-31  667818864766.2584
2026-09-30  685102503440.1578
2026-10-31  696794499724.2833
2026-11-30   759272490947.156
2026-12-31  781563189680.0571
2027-01-31  802517921411.4166
2027-02-28  728479774632.7329
2027-03-31  754360356470.6481
2027-04-30  767368759042.6244
2027-05-31  778333831342.8925
